# Conjunto de datos completo sin clusterización

In [1]:
#Importaciones
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler

#Lectura de datos
datos = pd.read_excel('03_Clusterizacion.xlsx')
datos.head(24)

,Fecha,Generación,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Cluster KMeans,Cluster GMM
0,2022-09-01 00:00:00,0.000000,19,7,77,0,4,15,0,Noche,Noche
1,2022-09-01 01:00:00,0.000000,19,7,82,0,4,16,1,Noche,Noche
2,2022-09-01 02:00:00,0.000000,18,9,85,0,3,16,2,Noche,Noche
3,2022-09-01 03:00:00,0.000000,18,11,87,0,3,16,3,Noche,Noche
4,2022-09-01 04:00:00,0.000000,18,11,88,0,3,16,4,Noche,Noche
5,2022-09-01 05:00:00,0.000000,17,15,86,0,3,14,5,Noche,Noche
6,2022-09-01 06:00:00,0.000000,18,47,89,0,3,16,6,Nublado,Lluvioso
7,2022-09-01 07:00:00,6.584959,18,51,95,0,4,17,7,Nublado,Lluvioso
8,2022-09-01 08:00:00,560.422022,18,47,100,0,3,18,8,Nublado,Lluvioso
9,2022-09-01 09:00:00,7720.582326,18,5,100,1,4,18,9,Nublado,Lluvioso


In [2]:
datos["Generacion_prev_hour"] = datos["Generación"].shift(1)
datos["Generacion_prev_day"] = datos["Generación"].shift(24)
datos = datos.dropna(how="any", axis= 0)

Definimos X y y

In [3]:
datos_dia = datos[datos["Cluster KMeans"] == "Nublado"].copy()
datos_dia.head(10)

,Fecha,Generación,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Cluster KMeans,Cluster GMM,Generacion_prev_hour,Generacion_prev_day
30,2022-09-02 06:00:00,0.000000,17,7,91,0,4,15,6,Nublado,Lluvioso,0.000000,0.000000
31,2022-09-02 07:00:00,0.000000,17,7,94,0,3,16,7,Nublado,Lluvioso,0.000000,6.584959
32,2022-09-02 08:00:00,438.814997,16,5,97,0,3,15,8,Nublado,Lluvioso,0.000000,560.422022
33,2022-09-02 09:00:00,5908.000884,17,0,93,1,2,16,9,Nublado,Lluvioso,438.814997,7720.582326
34,2022-09-02 10:00:00,5030.740421,18,0,85,2,2,15,10,Nublado,Lluvioso,5908.000884,9433.109309
39,2022-09-02 15:00:00,24647.568577,26,7,33,5,4,8,15,Nublado,Lluvioso,28500.000000,30000.000000
40,2022-09-02 16:00:00,25500.000000,27,7,34,4,4,9,16,Nublado,Lluvioso,24647.568577,28062.328964
41,2022-09-02 17:00:00,24281.956494,28,7,36,2,4,12,17,Nublado,Lluvioso,25500.000000,28786.629243
42,2022-09-02 18:00:00,22733.515002,26,7,39,1,4,12,18,Nublado,Lluvioso,24281.956494,29900.303971
43,2022-09-02 19:00:00,11972.590689,25,7,44,1,4,12,19,Nublado,Lluvioso,22733.515002,18282.505369


In [4]:
columns = datos_dia.drop(columns=["Fecha", "Generación", "Cluster KMeans", "Cluster GMM"]).columns

In [5]:
X = datos_dia[columns]
X

,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Generacion_prev_hour,Generacion_prev_day
30,17,7,91,0,4,15,6,0.000000,0.000000
31,17,7,94,0,3,16,7,0.000000,6.584959
32,16,5,97,0,3,15,8,0.000000,560.422022
33,17,0,93,1,2,16,9,438.814997,7720.582326
34,18,0,85,2,2,15,10,5908.000884,9433.109309
...,...,...,...,...,...,...,...,...,...
18273,14,0,87,1,4,11,8,67.000000,7302.000000
18274,15,0,83,2,4,12,9,7356.000000,18014.000000
18275,17,0,71,4,3,12,10,17638.000000,23010.000000
18276,19,0,60,5,3,11,11,23339.000000,26156.000000


In [6]:
y = datos_dia[['Generación']]
y

,Generación
30,0.000000
31,0.000000
32,438.814997
33,5908.000884
34,5030.740421
...,...
18273,7356.000000
18274,17638.000000
18275,23339.000000
18276,26323.000000


Dividimos entrenamiento, validación y prueba

In [7]:
train_size = int(0.7 * len(X))
val_size = int(0.85 * len(X))

In [8]:
# Entrenamiento, validación y prueba, 75, 15 y 15
X_train, y_train =  X.iloc[:train_size, :], y.iloc[:train_size, :]
X_val, y_val = X.iloc[train_size:val_size, :], y.iloc[train_size:val_size, :]
X_test, y_test = X.iloc[val_size:, :],  y.iloc[val_size:,:]

print(f'X_train: {len(X_train)}, y_train: {len(y_train)}')
print(f'X_val: {len(X_val)}, y_val: {len(y_val)}')
print(f'X_test: {len(X_test)}, y_test: {len(y_test)}')

X_train: 3649, y_train: 3649
X_val: 782, y_val: 782
X_test: 783, y_test: 783


## Escalar con MinMaxScaler

In [9]:
from sklearn.preprocessing import MinMaxScaler

In [10]:
x_scaler = MinMaxScaler().fit(X_train)
x_scaler

MinMaxScaler()

In [11]:
X_train_scaled = x_scaler.transform(X_train)
print(X_train_scaled)
print(X_train_scaled.shape)

[[5.00000000e-01 7.77777778e-02 9.04255319e-01 ... 0.00000000e+00
  0.00000000e+00 0.00000000e+00]
 [5.00000000e-01 7.77777778e-02 9.36170213e-01 ... 6.66666667e-02
  0.00000000e+00 2.19498623e-04]
 [4.68750000e-01 5.55555556e-02 9.68085106e-01 ... 1.33333333e-01
  0.00000000e+00 1.86807341e-02]
 ...
 [2.50000000e-01 1.11111111e-02 9.78723404e-01 ... 6.66666667e-02
  0.00000000e+00 0.00000000e+00]
 [2.18750000e-01 0.00000000e+00 1.00000000e+00 ... 1.33333333e-01
  0.00000000e+00 5.96666667e-03]
 [1.87500000e-01 0.00000000e+00 1.00000000e+00 ... 2.00000000e-01
  9.83333333e-03 3.62200000e-01]]
(3649, 9)


In [12]:
X_train_scaled_df = pd.DataFrame(X_train_scaled, index=X_train.index, columns=X_train.columns)
X_train_scaled_df

,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Generacion_prev_hour,Generacion_prev_day
30,0.50000,0.077778,0.904255,0.000000,0.666667,0.789474,0.000000,0.000000,0.000000
31,0.50000,0.077778,0.936170,0.000000,0.333333,0.842105,0.066667,0.000000,0.000219
32,0.46875,0.055556,0.968085,0.000000,0.333333,0.789474,0.133333,0.000000,0.018681
33,0.50000,0.000000,0.925532,0.083333,0.000000,0.842105,0.200000,0.014627,0.257353
34,0.53125,0.000000,0.840426,0.166667,0.000000,0.789474,0.266667,0.196933,0.314437
...,...,...,...,...,...,...,...,...,...
12958,0.53125,0.000000,0.446809,0.000000,0.333333,0.368421,1.000000,0.000000,0.000000
12967,0.28125,0.011111,0.978723,0.000000,1.000000,0.473684,0.000000,0.000000,0.000000
12968,0.25000,0.011111,0.978723,0.000000,1.000000,0.473684,0.066667,0.000000,0.000000
12969,0.21875,0.000000,1.000000,0.000000,0.333333,0.421053,0.133333,0.000000,0.005967


In [13]:
X_val_scaled = x_scaler.transform(X_val)
print(X_val_scaled)
print(X_val_scaled.shape)

[[0.28125    0.         0.82978723 ... 0.26666667 0.31593333 0.63086667]
 [0.375      0.         0.67021277 ... 0.33333333 0.65066667 0.66426667]
 [0.46875    0.         0.54255319 ... 0.4        0.67886667 0.6496    ]
 ...
 [0.75       0.02222222 0.43617021 ... 0.86666667 0.34693333 0.1592    ]
 [0.6875     0.06666667 0.5106383  ... 0.93333333 0.20373333 0.0115    ]
 [0.46875    0.07777778 0.87234043 ... 0.         0.         0.        ]]
(782, 9)


In [14]:
X_val_scaled_df = pd.DataFrame(X_val_scaled, index=X_val.index, columns=X_val.columns)
X_val_scaled_df

,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Generacion_prev_hour,Generacion_prev_day
12971,0.28125,0.000000,0.829787,0.166667,0.333333,0.368421,0.266667,0.315933,0.630867
12972,0.37500,0.000000,0.670213,0.166667,0.333333,0.368421,0.333333,0.650667,0.664267
12973,0.46875,0.000000,0.542553,0.250000,0.333333,0.368421,0.400000,0.678867,0.649600
12974,0.53125,0.000000,0.457447,0.250000,0.333333,0.368421,0.466667,0.656667,0.798233
12981,0.53125,0.000000,0.436170,0.000000,0.666667,0.368421,0.933333,0.042467,0.000000
...,...,...,...,...,...,...,...,...,...
16383,0.78125,0.000000,0.382979,0.750000,0.666667,0.631579,0.533333,0.900000,0.362167
16387,0.78125,0.000000,0.382979,0.166667,0.666667,0.631579,0.800000,0.379967,0.348467
16388,0.75000,0.022222,0.436170,0.000000,1.000000,0.684211,0.866667,0.346933,0.159200
16389,0.68750,0.066667,0.510638,0.000000,0.333333,0.684211,0.933333,0.203733,0.011500


In [15]:
X_test_scaled = x_scaler.transform(X_test)
print(X_test_scaled)
print(X_test_scaled.shape)

[[0.46875    0.06666667 0.87234043 ... 0.06666667 0.         0.06736667]
 [0.46875    0.02222222 0.86170213 ... 0.13333333 0.0214     0.4988    ]
 [0.5        0.02222222 0.80851064 ... 0.2        0.2268     0.8451    ]
 ...
 [0.5        0.         0.69148936 ... 0.26666667 0.58793333 0.767     ]
 [0.5625     0.         0.57446809 ... 0.33333333 0.77796667 0.87186667]
 [0.6875     0.         0.40425532 ... 0.46666667 0.8759     0.8551    ]]
(783, 9)


In [16]:
X_test_scaled_df = pd.DataFrame(X_test_scaled, index=X_test.index, columns=X_test.columns)
X_test_scaled_df

,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Generacion_prev_hour,Generacion_prev_day
16400,0.46875,0.066667,0.872340,0.000000,1.000000,0.736842,0.066667,0.000000,0.067367
16401,0.46875,0.022222,0.861702,0.083333,1.000000,0.736842,0.133333,0.021400,0.498800
16402,0.50000,0.022222,0.808511,0.166667,1.000000,0.736842,0.200000,0.226800,0.845100
16403,0.59375,0.022222,0.702128,0.166667,1.000000,0.736842,0.266667,0.350033,0.895933
16404,0.65625,0.022222,0.606383,0.583333,0.333333,0.736842,0.333333,0.438233,0.900000
...,...,...,...,...,...,...,...,...,...
18273,0.40625,0.000000,0.861702,0.083333,0.666667,0.578947,0.133333,0.002233,0.243400
18274,0.43750,0.000000,0.819149,0.166667,0.666667,0.631579,0.200000,0.245200,0.600467
18275,0.50000,0.000000,0.691489,0.333333,0.333333,0.631579,0.266667,0.587933,0.767000
18276,0.56250,0.000000,0.574468,0.416667,0.333333,0.578947,0.333333,0.777967,0.871867


In [17]:
x_scaller_all = MinMaxScaler().fit(X)
print(x_scaller_all)

MinMaxScaler()


In [18]:
X_scaled = x_scaller_all.transform(X)
print(X_scaled)
print(X_scaled.shape)

[[4.32432432e-01 7.77777778e-02 9.06250000e-01 ... 0.00000000e+00
  0.00000000e+00 0.00000000e+00]
 [4.32432432e-01 7.77777778e-02 9.37500000e-01 ... 6.66666667e-02
  0.00000000e+00 2.19498623e-04]
 [4.05405405e-01 5.55555556e-02 9.68750000e-01 ... 1.33333333e-01
  0.00000000e+00 1.86807341e-02]
 ...
 [4.32432432e-01 0.00000000e+00 6.97916667e-01 ... 2.66666667e-01
  5.87933333e-01 7.67000000e-01]
 [4.86486486e-01 0.00000000e+00 5.83333333e-01 ... 3.33333333e-01
  7.77966667e-01 8.71866667e-01]
 [5.94594595e-01 0.00000000e+00 4.16666667e-01 ... 4.66666667e-01
  8.75900000e-01 8.55100000e-01]]
(5214, 9)


In [19]:
X_scaled_df = pd.DataFrame(X_scaled, index=X.index, columns=X.columns)
X_scaled_df

,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Generacion_prev_hour,Generacion_prev_day
30,0.432432,0.077778,0.906250,0.000000,0.666667,0.75,0.000000,0.000000,0.000000
31,0.432432,0.077778,0.937500,0.000000,0.333333,0.80,0.066667,0.000000,0.000219
32,0.405405,0.055556,0.968750,0.000000,0.333333,0.75,0.133333,0.000000,0.018681
33,0.432432,0.000000,0.927083,0.083333,0.000000,0.80,0.200000,0.014627,0.257353
34,0.459459,0.000000,0.843750,0.166667,0.000000,0.75,0.266667,0.196933,0.314437
...,...,...,...,...,...,...,...,...,...
18273,0.351351,0.000000,0.864583,0.083333,0.666667,0.55,0.133333,0.002233,0.243400
18274,0.378378,0.000000,0.822917,0.166667,0.666667,0.60,0.200000,0.245200,0.600467
18275,0.432432,0.000000,0.697917,0.333333,0.333333,0.60,0.266667,0.587933,0.767000
18276,0.486486,0.000000,0.583333,0.416667,0.333333,0.55,0.333333,0.777967,0.871867


In [20]:
y_scaler = MinMaxScaler().fit(y_train)
print(y_scaler)

MinMaxScaler()


In [21]:
y_train_scaled = y_scaler.transform(y_train)
print(y_train_scaled)
print(y_train_scaled.shape)

[[0.        ]
 [0.        ]
 [0.01462717]
 ...
 [0.        ]
 [0.00983333]
 [0.31593333]]
(3649, 1)


In [22]:
y_train_scaled_df = pd.DataFrame(y_train_scaled, index=y_train.index, columns=y_train.columns)
y_train_scaled_df

,Generación
30,0.000000
31,0.000000
32,0.014627
33,0.196933
34,0.167691
...,...
12958,0.000000
12967,0.000000
12968,0.000000
12969,0.009833


In [23]:
y_val_scaled = y_scaler.transform(y_val)
print(y_val_scaled)
print(y_val_scaled.shape)

[[6.50666667e-01]
 [6.78866667e-01]
 [6.56666667e-01]
 [6.34033333e-01]
 [0.00000000e+00]
 [0.00000000e+00]
 [0.00000000e+00]
 [0.00000000e+00]
 [3.17000000e-02]
 [4.02500000e-01]
 [6.29733333e-01]
 [6.02466667e-01]
 [3.92200000e-01]
 [4.24333333e-02]
 [0.00000000e+00]
 [0.00000000e+00]
 [0.00000000e+00]
 [3.01333333e-02]
 [3.76966667e-01]
 [7.10300000e-01]
 [7.51233333e-01]
 [7.85666667e-01]
 [0.00000000e+00]
 [0.00000000e+00]
 [0.00000000e+00]
 [0.00000000e+00]
 [0.00000000e+00]
 [1.57333333e-02]
 [4.93300000e-01]
 [8.19466667e-01]
 [8.34700000e-01]
 [8.20833333e-01]
 [7.17400000e-01]
 [8.18666667e-01]
 [6.90233333e-01]
 [4.11033333e-01]
 [4.34666667e-02]
 [0.00000000e+00]
 [0.00000000e+00]
 [0.00000000e+00]
 [7.32000000e-02]
 [5.93533333e-01]
 [6.08633333e-01]
 [8.79666667e-01]
 [6.04733333e-01]
 [1.12600000e-01]
 [0.00000000e+00]
 [0.00000000e+00]
 [0.00000000e+00]
 [0.00000000e+00]
 [1.32066667e-01]
 [2.87266667e-01]
 [8.49266667e-01]
 [8.96300000e-01]
 [0.00000000e+00]
 [0.000000

In [24]:
y_val_scaled_df = pd.DataFrame(y_val_scaled, index=y_val.index, columns=y_val.columns)
y_val_scaled_df

,Generación
12971,0.650667
12972,0.678867
12973,0.656667
12974,0.634033
12981,0.000000
...,...
16383,0.804800
16387,0.346933
16388,0.203733
16389,0.014900


In [25]:
y_test_scaled = y_scaler.transform(y_test)
print(y_test_scaled)
print(y_test_scaled.shape)

[[2.14000000e-02]
 [2.26800000e-01]
 [3.50033333e-01]
 [4.38233333e-01]
 [5.17866667e-01]
 [2.55666667e-02]
 [0.00000000e+00]
 [0.00000000e+00]
 [2.79666667e-02]
 [2.78633333e-01]
 [4.04633333e-01]
 [3.96600000e-01]
 [3.95833333e-01]
 [3.86166667e-01]
 [5.20300000e-01]
 [4.55600000e-01]
 [3.93566667e-01]
 [2.66166667e-01]
 [2.36933333e-01]
 [2.46166667e-01]
 [2.48333333e-02]
 [0.00000000e+00]
 [0.00000000e+00]
 [2.20333333e-02]
 [2.95566667e-01]
 [4.52433333e-01]
 [5.00800000e-01]
 [4.54266667e-01]
 [2.79633333e-01]
 [2.82066667e-01]
 [2.80133333e-01]
 [2.70033333e-01]
 [4.37733333e-01]
 [2.42166667e-01]
 [2.97333333e-02]
 [0.00000000e+00]
 [0.00000000e+00]
 [2.84000000e-02]
 [2.61400000e-01]
 [5.28433333e-01]
 [5.05033333e-01]
 [3.92000000e-01]
 [4.46200000e-01]
 [8.93633333e-01]
 [7.99433333e-01]
 [3.90766667e-01]
 [4.55000000e-02]
 [0.00000000e+00]
 [0.00000000e+00]
 [3.93000000e-02]
 [3.93100000e-01]
 [7.95400000e-01]
 [9.24766667e-01]
 [0.00000000e+00]
 [4.36666667e-02]
 [4.353333

In [26]:
y_test_scaled_df = pd.DataFrame(y_test_scaled, index=y_test.index, columns=y_test.columns)
y_test_scaled_df

,Generación
16400,0.021400
16401,0.226800
16402,0.350033
16403,0.438233
16404,0.517867
...,...
18273,0.245200
18274,0.587933
18275,0.777967
18276,0.877433


In [27]:
y_scaller_all = MinMaxScaler().fit(y)
print(y_scaller_all)

MinMaxScaler()


In [28]:
y_scaled = y_scaller_all.transform(y)
print(y_scaled)
print(y_scaled.shape)

[[0.        ]
 [0.        ]
 [0.01462717]
 ...
 [0.77796667]
 [0.87743333]
 [0.85326667]]
(5214, 1)


In [29]:
y_scaled_df = pd.DataFrame(y_scaled, index=y.index, columns=y.columns)
y_scaled_df

,Generación
30,0.000000
31,0.000000
32,0.014627
33,0.196933
34,0.167691
...,...
18273,0.245200
18274,0.587933
18275,0.777967
18276,0.877433


## Definición de modelos

### RandomForest

In [30]:
from lightgbm import LGBMRegressor
import optuna
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score
import seaborn as sns
from sklearn.metrics import mean_absolute_percentage_error as mean_absolute_percentage_error
from sklearn.metrics import mean_absolute_error as mean_absolute_error
from sklearn.metrics import mean_squared_error as mean_squared_error
from sklearn.metrics import r2_score as r2_score

In [31]:
# Inicializar listas para métricas
LightGBM_model = LGBMRegressor(num_leaves=500, subsample= 0.10698460631792395, colsample_bytree= 0.7272836809565294, min_data_in_leaf= 85)
LightGBM_model.fit(X_train_scaled_df, y_train_scaled_df)
resultados = pd.DataFrame(index = y_test_scaled_df.index, columns=["LightGBM"])
#Ciclo diario de predicción
for i in range(len(X_test)):
    inicio = i * 1
    fin = inicio + 1

    X_test_seg = X_test_scaled_df.iloc[inicio:fin, :]
    y_test_seg = y_test_scaled_df.iloc[inicio:fin]

    if len(X_test_seg) < 1:
        break

    y_pred = LightGBM_model.predict(X_test_seg)
    y_pred = y_scaler.inverse_transform(y_pred.reshape(-1, 1))
    y_pred = np.clip(y_pred, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000

    resultados.iloc[i, 0] = y_pred[0, 0]

[LightGBM] [Warning] min_data_in_leaf is set=85, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=85
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Warning] min_data_in_leaf is set=85, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=85
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.177061 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 747
[LightGBM] [Info] Number of data points in the train set: 3649, number of used features: 9
[LightGBM] [Info] Start training from score 0.295448
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further split

In [32]:
resultados

,LightGBM
16400,1485.421686
16401,11718.606066
16402,18614.042112
16403,17689.00615
16404,17039.389972
...,...
18273,6900.654023
18274,20543.60541
18275,23189.371101
18276,26104.467998


In [33]:
predicciones = y_test.copy()
predicciones

,Generación
16400,642.0
16401,6804.0
16402,10501.0
16403,13147.0
16404,15536.0
...,...
18273,7356.0
18274,17638.0
18275,23339.0
18276,26323.0


In [34]:
predicciones["LightGBM"] = resultados["LightGBM"]
predicciones

,Generación,LightGBM
16400,642.0,1485.421686
16401,6804.0,11718.606066
16402,10501.0,18614.042112
16403,13147.0,17689.00615
16404,15536.0,17039.389972
...,...,...
18273,7356.0,6900.654023
18274,17638.0,20543.60541
18275,23339.0,23189.371101
18276,26323.0,26104.467998


In [35]:
print(f"MAE: {mean_absolute_error(predicciones['Generación'], predicciones['LightGBM']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones['Generación'], predicciones['LightGBM'])):.4f}")
print(f"R²: {r2_score(predicciones['Generación'], predicciones['LightGBM']):.4f}")

MAE: 1424.1067
RMSE: 2418.9925
R²: 0.9503


## Random Forest

In [36]:
from sklearn.ensemble import RandomForestRegressor

In [37]:
#Modelo LightGBM
RF_model = RandomForestRegressor(
    criterion="squared_error",
    random_state=0,
    n_estimators=400,
    min_impurity_decrease=0,
    max_depth=None,
    bootstrap=True
)
RF_model.fit(X_train_scaled_df, y_train_scaled_df)
# Inicializar listas para métricas
resultados = pd.DataFrame(index = y_test_scaled_df.index, columns=["Random Forest"])
#Ciclo diario de predicción
for i in range(len(X_test)):
    inicio = i * 1
    fin = inicio + 1

    X_test_seg = X_test_scaled_df.iloc[inicio:fin, :]
    y_test_seg = y_test_scaled_df.iloc[inicio:fin]

    if len(X_test_seg) < 1:
        break

    y_pred = RF_model.predict(X_test_seg)
    y_pred = y_scaler.inverse_transform(y_pred.reshape(-1, 1))
    y_pred = np.clip(y_pred, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000

    resultados.iloc[i, 0] = y_pred[0, 0]

In [38]:
predicciones["Random Forest"] = resultados["Random Forest"]
predicciones

,Generación,LightGBM,Random Forest
16400,642.0,1485.421686,1850.979413
16401,6804.0,11718.606066,11182.089488
16402,10501.0,18614.042112,19279.054526
16403,13147.0,17689.00615,17438.807493
16404,15536.0,17039.389972,16974.916203
...,...,...,...
18273,7356.0,6900.654023,5846.106045
18274,17638.0,20543.60541,19516.687678
18275,23339.0,23189.371101,21231.472263
18276,26323.0,26104.467998,24456.544978


## Preparación redes neuronales

In [39]:
import numpy as np
import pandas as pd

def create_sliding_window_with_index(data_X, data_y, lookback):
    X, y, indices = [], [], []
    
    # Asegurar que `data_y` tiene los mismos índices que `data_X`
    data_y = data_y.reindex(data_X.index)

    max_index = len(data_X) - lookback

    for i in range(max_index):
        X.append(data_X.iloc[i:i + lookback].values)  # Ventana de entrada
        
        # Obtener el índice correcto en `data_y`
        y_index = data_X.index[i + lookback]

        # Extraer el valor correspondiente de `data_y`
        if y_index in data_y.index:
            y_value = data_y.loc[y_index]
            if isinstance(y_value, pd.Series):  # Si devuelve una serie, extraer el valor
                y_value = y_value.iloc[0]
        else:
            y_value = np.nan  # Si no está, asignamos NaN

        y.append(y_value)
        indices.append(y_index)  # 🔹 Guardamos el índice original de `data_y`

    # Convertimos `X` en un array y `y` en DataFrame conservando sus índices originales
    X_array = np.array(X)
    y_df = pd.DataFrame(y, index=indices, columns=['y'])  # 🔹 Conservamos los índices originales

    return X_array, y_df


In [40]:
lookback = 48  # Puedes ajustar a 24, 72, etc.

# Aplicar la ventana deslizante a cada conjunto
X_train_windowed, y_train_windowed = create_sliding_window_with_index(X_train_scaled_df, y_train_scaled_df, lookback)
X_val_windowed, y_val_windowed = create_sliding_window_with_index(X_val_scaled_df, y_val_scaled_df, lookback)
X_test_windowed, y_test_windowed = create_sliding_window_with_index(X_test_scaled_df, y_test_scaled_df, lookback)


In [41]:
print(f'X_train: {X_train_windowed.shape}, y_train: {y_train_windowed.shape}')
print(f'X_val: {X_val_windowed.shape}, y_val: {y_val_windowed.shape}')
print(f'X_test: {X_test_windowed.shape}, y_test: {y_test_windowed.shape}')

X_train: (3601, 48, 9), y_train: (3601, 1)
X_val: (734, 48, 9), y_val: (734, 1)
X_test: (735, 48, 9), y_test: (735, 1)


## CTNET

In [42]:
import tensorflow as tf
from tensorflow.keras import layers

In [43]:
def compile_and_fit(model, xtrain=X_train_windowed, ytrain=y_train_windowed, learning_rate=0.0001):
    model.compile(loss=[tf.keras.losses.MeanSquaredError()],
                  optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
                  metrics=[tf.keras.metrics.RootMeanSquaredError(), tf.keras.metrics.MeanAbsolutePercentageError(), tf.keras.metrics.MeanAbsoluteError()])
    
    history = model.fit(xtrain, ytrain, epochs=50,
                        batch_size=512, validation_split=0.2, verbose=1)
    return history

def Loss(train_loss, valid_loss):
    plt.plot(train_loss)
    plt.plot(valid_loss)
    plt.rcParams["figure.figsize"] = (15, 3)
    plt.title('Model Losses')
    plt.ylabel('Loss')
    plt.xlabel('Epoch')
    plt.legend(['Train Loss', 'Validation Loss'], loc='upper left')
    plt.savefig('out/loss_plot.png')
    plt.show()

def transformer_encoder(inputs, head_size, num_heads, ff_dim, dropout=0):
    x = layers.LayerNormalization()(inputs)
    x = layers.Conv1D(filters=ff_dim, kernel_size=1, activation="relu", padding="same")(x)
    x = layers.Conv1D(filters=128, kernel_size=2, activation="relu", padding="same")(x)
    x = layers.Conv1D(filters=inputs.shape[-1], kernel_size=1)(x)
    res = x + inputs
    norm_x = layers.LayerNormalization()(res)
    x = layers.MultiHeadAttention(key_dim=head_size, num_heads=num_heads, dropout=dropout)(norm_x, norm_x)
    res = x + inputs
    norm_x = layers.LayerNormalization()(res)
    return norm_x

def build_model(input_shape, head_size, num_heads, ff_dim, num_transformer_blocks, mlp_units, dropout=0, mlp_dropout=0):
    inputs = tf.keras.Input(shape=input_shape)
    x = inputs
    
    for _ in range(num_transformer_blocks):
        enc_out = transformer_encoder(x, head_size, num_heads, ff_dim, dropout)
    
    x = layers.MultiHeadAttention(key_dim=head_size, num_heads=num_heads, dropout=dropout)(enc_out, enc_out)
    res = x + enc_out
    x = layers.LayerNormalization(epsilon=1e-6)(res)
    x = layers.GlobalAveragePooling1D(data_format="channels_first")(x)
    x = layers.Dense(832, activation="relu")(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(mlp_dropout)(x)

    outputs = layers.Dense(1)(x)
    
    return tf.keras.Model(inputs, outputs)

In [44]:
CTNET = build_model((X_train_windowed.shape[1], X_train_windowed.shape[2]), head_size=4, num_heads=3, ff_dim=32, num_transformer_blocks=3, mlp_units=[256], mlp_dropout=0.3, dropout=0.2)

In [45]:
history = compile_and_fit(CTNET)

Epoch 1/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 99s 3s/step - loss: 0.1892 - mean_absolute_error: 0.2915 - mean_absolute_percentage_error: 480896.1875 - root_mean_squared_error: 0.4349 - val_loss: 0.1760 - val_mean_absolute_error: 0.2925 - val_mean_absolute_percentage_error: 2641551.5000 - val_root_mean_squared_error: 0.4196
Epoch 2/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 9s 1s/step - loss: 0.1829 - mean_absolute_error: 0.2871 - mean_absolute_percentage_error: 3112354.2500 - root_mean_squared_error: 0.4277 - val_loss: 0.1700 - val_mean_absolute_error: 0.2904 - val_mean_absolute_percentage_error: 6072492.0000 - val_root_mean_squared_error: 0.4123
Epoch 3/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 8s 761ms/step - loss: 0.1801 - mean_absolute_error: 0.2868 - mean_absolute_percentage_error: 6482014.0000 - root_mean_squared_error: 0.4244 - val_loss: 0.1628 - val_mean_absolute_error: 0.2882 - val_mean_absolute_percentage_error: 10329943.0000 - val_root_mean_squared_error: 0.4035
Epoch 4/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 7s 685ms/step -

In [46]:
CTNET_predictions = CTNET.predict(X_test_windowed)
CTNET_predictions

23/23 ━━━━━━━━━━━━━━━━━━━━ 14s 357ms/step


array([[0.20636004],
       [0.21218029],
       [0.21842426],
       [0.23275986],
       [0.2622364 ],
       [0.26994184],
       [0.2747161 ],
       [0.27569607],
       [0.2718992 ],
       [0.27850738],
       [0.29675904],
       [0.29263696],
       [0.28222615],
       [0.2900171 ],
       [0.3074705 ],
       [0.3153673 ],
       [0.31506833],
       [0.3102987 ],
       [0.29964423],
       [0.27214795],
       [0.25492063],
       [0.2480689 ],
       [0.24975416],
       [0.2542193 ],
       [0.2717655 ],
       [0.30526367],
       [0.33614632],
       [0.36833432],
       [0.35459152],
       [0.3508117 ],
       [0.3559769 ],
       [0.3501242 ],
       [0.36101225],
       [0.38539514],
       [0.40035108],
       [0.41472885],
       [0.41750035],
       [0.41883606],
       [0.39787132],
       [0.37962183],
       [0.37038314],
       [0.35004583],
       [0.3481113 ],
       [0.34927365],
       [0.35650998],
       [0.36301   ],
       [0.39317036],
       [0.417

In [47]:
CTNET_predictions = y_scaler.inverse_transform(CTNET_predictions.reshape(-1, 1))
CTNET_predictions = np.clip(CTNET_predictions, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000

In [48]:
resultados = pd.DataFrame(CTNET_predictions, index = y_test_windowed.index, columns=["CTNET"])

In [49]:
predicciones["CTNET"] = resultados["CTNET"]
predicciones

,Generación,LightGBM,Random Forest,CTNET
16400,642.0,1485.421686,1850.979413,NaN
16401,6804.0,11718.606066,11182.089488,NaN
16402,10501.0,18614.042112,19279.054526,NaN
16403,13147.0,17689.00615,17438.807493,NaN
16404,15536.0,17039.389972,16974.916203,NaN
...,...,...,...,...
18273,7356.0,6900.654023,5846.106045,9994.365234
18274,17638.0,20543.60541,19516.687678,10577.663086
18275,23339.0,23189.371101,21231.472263,11426.608398
18276,26323.0,26104.467998,24456.544978,12110.069336


In [50]:
predicciones["CTNET"] = predicciones["CTNET"].fillna(0)

In [51]:
# import optuna
# import tensorflow as tf
# from tensorflow.keras import layers
# from sklearn.model_selection import train_test_split

# # Definir la función objetivo para Optuna
# def objective(trial):
#     # Sugerir valores para los hiperparámetros
#     head_size = trial.suggest_int("head_size", 8, 64, step=8)
#     num_heads = trial.suggest_int("num_heads", 2, 8, step=2)
#     ff_dim = trial.suggest_int("ff_dim", 32, 256, step=32)
#     num_transformer_blocks = trial.suggest_int("num_transformer_blocks", 1, 4)
#     mlp_units = trial.suggest_categorical("mlp_units", [[128, 64], [256, 128, 64], [512, 256, 128]])
#     dropout = trial.suggest_float("dropout", 0.1, 0.5, step=0.1)
#     mlp_dropout = trial.suggest_float("mlp_dropout", 0.1, 0.5, step=0.1)
#     learning_rate = trial.suggest_loguniform("learning_rate", 1e-5, 1e-2)

#     # Construcción del modelo con los hiperparámetros sugeridos
#     model = build_model(
#         input_shape=X_train_windowed.shape[1:],
#         head_size=head_size,
#         num_heads=num_heads,
#         ff_dim=ff_dim,
#         num_transformer_blocks=num_transformer_blocks,
#         mlp_units=mlp_units,
#         dropout=dropout,
#         mlp_dropout=mlp_dropout
#     )

#     # Compilar el modelo con los hiperparámetros sugeridos
#     model.compile(
#         loss=tf.keras.losses.MeanSquaredError(),
#         optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
#         metrics=[tf.keras.metrics.RootMeanSquaredError()]
#     )

#     # Entrenamiento con un número reducido de épocas para acelerar la búsqueda
#     history = model.fit(
#         X_train_windowed, y_train_windowed,
#         validation_split=0.2,
#         epochs=50,  # Reducimos las épocas para acelerar la búsqueda
#         batch_size=512,
#         verbose=0
#     )

#     # Obtener la métrica de validación (RMSE) y minimizarla
#     val_rmse = min(history.history["val_root_mean_squared_error"])
    
#     return val_rmse  # Queremos minimizar el RMSE

# # Ejecutar la optimización de hiperparámetros
# study = optuna.create_study(direction="minimize")
# study.optimize(objective, n_trials=20, timeout=3600)  # 20 iteraciones, máximo 1 hora

# # Mostrar los mejores hiperparámetros encontrados
# best_params = study.best_params
# print(f"Mejores hiperparámetros: {best_params}")


In [52]:
CTNET = build_model((X_train_windowed.shape[1], X_train_windowed.shape[2]), head_size=16, num_heads=8, ff_dim=256, num_transformer_blocks=1, mlp_units=[128,64], mlp_dropout=0.2, dropout=0.2)

In [53]:
history = compile_and_fit(CTNET, learning_rate = 0.0010762230908145116)

Epoch 1/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 120s 6s/step - loss: 0.1741 - mean_absolute_error: 0.2840 - mean_absolute_percentage_error: 8652461.0000 - root_mean_squared_error: 0.4171 - val_loss: 0.1036 - val_mean_absolute_error: 0.2757 - val_mean_absolute_percentage_error: 62603792.0000 - val_root_mean_squared_error: 0.3218
Epoch 2/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 36s 4s/step - loss: 0.1165 - mean_absolute_error: 0.2947 - mean_absolute_percentage_error: 81631552.0000 - root_mean_squared_error: 0.3413 - val_loss: 0.0972 - val_mean_absolute_error: 0.2881 - val_mean_absolute_percentage_error: 112659272.0000 - val_root_mean_squared_error: 0.3118
Epoch 3/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 43s 4s/step - loss: 0.1097 - mean_absolute_error: 0.2949 - mean_absolute_percentage_error: 89104056.0000 - root_mean_squared_error: 0.3312 - val_loss: 0.0983 - val_mean_absolute_error: 0.2759 - val_mean_absolute_percentage_error: 72689152.0000 - val_root_mean_squared_error: 0.3136
Epoch 4/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 21s 3s/st

In [54]:
CTNET_predictions = CTNET.predict(X_test_windowed)
CTNET_predictions

23/23 ━━━━━━━━━━━━━━━━━━━━ 24s 429ms/step


array([[0.01158301],
       [0.01909353],
       [0.10304229],
       [0.5667744 ],
       [0.57705253],
       [0.82304674],
       [0.01873512],
       [0.1178408 ],
       [0.60313505],
       [0.70437944],
       [0.7857744 ],
       [0.80544645],
       [0.20825447],
       [0.6573958 ],
       [0.7080984 ],
       [0.709789  ],
       [0.7446922 ],
       [0.7440092 ],
       [0.5094593 ],
       [0.0256865 ],
       [0.01241064],
       [0.01837815],
       [0.01820627],
       [0.12468305],
       [0.64880735],
       [0.6628656 ],
       [0.735369  ],
       [0.80515605],
       [0.06171402],
       [0.02783627],
       [0.03151066],
       [0.18363217],
       [0.64018166],
       [0.65750664],
       [0.60417074],
       [0.5286783 ],
       [0.48038572],
       [0.49012214],
       [0.23361686],
       [0.31374952],
       [0.43672037],
       [0.4347693 ],
       [0.49195656],
       [0.2677983 ],
       [0.09438153],
       [0.06850504],
       [0.03269611],
       [0.129

In [55]:
CTNET_predictions = y_scaler.inverse_transform(CTNET_predictions.reshape(-1, 1))
CTNET_predictions = np.clip(CTNET_predictions, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000

In [56]:
resultados = pd.DataFrame(CTNET_predictions, index = y_test_windowed.index, columns=["CTNET"])

In [57]:
predicciones["CTNET"] = resultados["CTNET"]
predicciones

,Generación,LightGBM,Random Forest,CTNET
16400,642.0,1485.421686,1850.979413,NaN
16401,6804.0,11718.606066,11182.089488,NaN
16402,10501.0,18614.042112,19279.054526,NaN
16403,13147.0,17689.00615,17438.807493,NaN
16404,15536.0,17039.389972,16974.916203,NaN
...,...,...,...,...
18273,7356.0,6900.654023,5846.106045,712.266174
18274,17638.0,20543.60541,19516.687678,16165.256836
18275,23339.0,23189.371101,21231.472263,20088.689453
18276,26323.0,26104.467998,24456.544978,24133.009766


## Forecasting

In [58]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import *
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.losses import MeanSquaredError
from tensorflow.keras.metrics import RootMeanSquaredError
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from tensorflow.keras.losses import Huber
from tensorflow.keras.callbacks import EarlyStopping

In [59]:
Forecast_model = Sequential()
Forecast_model.add(InputLayer((X_train_windowed.shape[1], X_train_windowed.shape[2])))

#CNN
Forecast_model.add(Conv1D(filters=64, kernel_size=2, padding='same', activation='relu'))
Forecast_model.add(BatchNormalization())  # 🔹 Nueva Normalización aquí
Forecast_model.add(MaxPooling1D(pool_size=2))

#model_Soleado.add(Flatten())
#BiLSTM
Forecast_model.add(Bidirectional(LSTM(128, return_sequences=True)))
Forecast_model.add(Bidirectional(LSTM(64, return_sequences=True)))
Forecast_model.add(Dropout(0.2))  # 🔹 Mayor regularización en BiLSTM
Forecast_model.add(Bidirectional(LSTM(32, return_sequences=False)))

#Normalización y Dropout
Forecast_model.add(BatchNormalization())
Forecast_model.add(Dropout(0.3))

# Capas Densas
Forecast_model.add(Dense(16, activation='relu'))
Forecast_model.add(Dense(1, 'relu'))

Forecast_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_12 (Conv1D)              │ (None, 48, 64)         │         1,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 48, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 24, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 24, 256)        │       197,632 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 24, 128)        │       164,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 24, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ (None, 64)             │        41,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 16)             │         1,040 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 405,985 (1.55 MB)

 Trainable params: 405,729 (1.55 MB)

 Non-trainable params: 256 (1.00 KB)

In [60]:
cp = ModelCheckpoint('Forcasting_model.keras', save_best_only=True)
Forecast_model.compile(optimizer=Adam(learning_rate=0.0001), loss=Huber(delta=1000), metrics=['mae'])
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

In [61]:
history = Forecast_model.fit(X_train_windowed, y_train_windowed, validation_data=(X_val_windowed, y_val_windowed), epochs=100, batch_size=8, callbacks=[cp, early_stop])

Epoch 1/100
451/451 ━━━━━━━━━━━━━━━━━━━━ 461s 487ms/step - loss: 0.1321 - mae: 0.3677 - val_loss: 0.1383 - val_mae: 0.3902
Epoch 2/100
451/451 ━━━━━━━━━━━━━━━━━━━━ 220s 383ms/step - loss: 0.0874 - mae: 0.2892 - val_loss: 0.1309 - val_mae: 0.3821
Epoch 3/100
451/451 ━━━━━━━━━━━━━━━━━━━━ 194s 354ms/step - loss: 0.0831 - mae: 0.2821 - val_loss: 0.1174 - val_mae: 0.3582
Epoch 4/100
451/451 ━━━━━━━━━━━━━━━━━━━━ 156s 331ms/step - loss: 0.0708 - mae: 0.2568 - val_loss: 0.0714 - val_mae: 0.2703
Epoch 5/100
451/451 ━━━━━━━━━━━━━━━━━━━━ 232s 388ms/step - loss: 0.0681 - mae: 0.2488 - val_loss: 0.0710 - val_mae: 0.2668
Epoch 6/100
451/451 ━━━━━━━━━━━━━━━━━━━━ 225s 429ms/step - loss: 0.0577 - mae: 0.2267 - val_loss: 0.0632 - val_mae: 0.2610
Epoch 7/100
451/451 ━━━━━━━━━━━━━━━━━━━━ 197s 409ms/step - loss: 0.0535 - mae: 0.2245 - val_loss: 0.0545 - val_mae: 0.2400
Epoch 8/100
451/451 ━━━━━━━━━━━━━━━━━━━━ 219s 436ms/step - loss: 0.0455 - mae: 0.1986 - val_loss: 0.0396 - val_mae: 0.2045
Epoch 9/100
451/

In [62]:
Forecast_predictions = Forecast_model.predict(X_test_windowed)
Forecast_predictions

23/23 ━━━━━━━━━━━━━━━━━━━━ 88s 2s/step 


array([[0.        ],
       [0.        ],
       [0.18055815],
       [0.5791993 ],
       [0.5102185 ],
       [0.3486672 ],
       [0.        ],
       [0.22459531],
       [0.5873128 ],
       [0.6484679 ],
       [0.4912983 ],
       [0.45978707],
       [0.        ],
       [0.5534802 ],
       [0.6566276 ],
       [0.75677526],
       [0.7758445 ],
       [0.6465111 ],
       [0.673817  ],
       [0.21723151],
       [0.        ],
       [0.        ],
       [0.07166737],
       [0.38640216],
       [0.6757587 ],
       [0.7122274 ],
       [0.74775827],
       [0.7120526 ],
       [0.        ],
       [0.        ],
       [0.        ],
       [0.28061748],
       [0.6326781 ],
       [0.6301483 ],
       [0.4875728 ],
       [0.63615584],
       [0.65196   ],
       [0.6787219 ],
       [0.55887586],
       [0.5657383 ],
       [0.36356255],
       [0.32341218],
       [0.30619344],
       [0.24874777],
       [0.        ],
       [0.        ],
       [0.        ],
       [0.180

In [63]:
Forecast_predictions = y_scaler.inverse_transform(Forecast_predictions.reshape(-1, 1))
Forecast_predictions = np.clip(Forecast_predictions, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000

In [64]:
Forecast_resultados = pd.DataFrame(Forecast_predictions, index = y_test_windowed.index, columns=["Forecast"])

In [65]:
predicciones["Forecast"] = Forecast_resultados["Forecast"]
predicciones

,Generación,LightGBM,Random Forest,CTNET,Forecast
16400,642.0,1485.421686,1850.979413,NaN,NaN
16401,6804.0,11718.606066,11182.089488,NaN,NaN
16402,10501.0,18614.042112,19279.054526,NaN,NaN
16403,13147.0,17689.00615,17438.807493,NaN,NaN
16404,15536.0,17039.389972,16974.916203,NaN,NaN
...,...,...,...,...,...
18273,7356.0,6900.654023,5846.106045,712.266174,11134.365234
18274,17638.0,20543.60541,19516.687678,16165.256836,17797.712891
18275,23339.0,23189.371101,21231.472263,20088.689453,18955.470703
18276,26323.0,26104.467998,24456.544978,24133.009766,14881.110352


## Métricas

In [66]:
predicciones.loc[~predicciones['CTNET'].isna(),'Generación']

16495        0.0
16496     1179.0
16497    11793.0
16498    23862.0
16500    27743.0
          ...   
18273     7356.0
18274    17638.0
18275    23339.0
18276    26323.0
18278    25598.0
Name: Generación, Length: 735, dtype: float64

## Photovoltaic

In [67]:
from tensorflow.keras.models import Model
inputs = Input(shape=(X_train_windowed.shape[1], X_train_windowed.shape[2]))

# Primera capa CNN
x = Conv1D(filters=64, kernel_size=4, padding='same', activation='relu')(inputs)
x = MaxPooling1D(pool_size=2)(x)

# Segunda capa CNN
x = Conv1D(filters=128, kernel_size=4, padding='same', activation='relu')(x)
x = MaxPooling1D(pool_size=2)(x)

# Capa BiGRU
x = Bidirectional(GRU(64, return_sequences=True))(x)

# Atención: se define de forma explícita
attention = MultiHeadAttention(num_heads=4, key_dim=128)(x, x)

# Aplanar y agregar Dropout
x = Flatten()(attention)
x = Dropout(0.4)(x)
initializer = tf.keras.initializers.HeNormal()
x = Dense(64, activation="relu", kernel_regularizer=l2(0.01))(x)
x = Dense(32, activation="relu")(x)  # Otra capa intermedia

# Capa de salida
outputs = Dense(1, activation="linear")(x)

# Definir el modelo
Photo_model = Model(inputs=inputs, outputs=outputs)

# Resumen del modelo
Photo_model.summary()

Model: "functional_13"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_3       │ (None, 48, 9)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_13 (Conv1D)  │ (None, 48, 64)    │      2,368 │ input_layer_3[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_1     │ (None, 24, 64)    │          0 │ conv1d_13[0][0]   │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_14 (Conv1D)  │ (None, 24, 128)   │     32,896 │ max_pooling1d_1[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_2     │ (None, 12, 128)   │          0 │ conv1d_14[0][0]   │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_3     │ (None, 12, 128)   │     74,496 │ max_pooling1d_2[… │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 12, 128)   │    263,808 │ bidirectional_3[… │
│ (MultiHeadAttentio… │                   │            │ bidirectional_3[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 1536)      │          0 │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_11          │ (None, 1536)      │          0 │ flatten[0][0]     │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_10 (Dense)    │ (None, 64)        │     98,368 │ dropout_11[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_11 (Dense)    │ (None, 32)        │      2,080 │ dense_10[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_12 (Dense)    │ (None, 1)         │         33 │ dense_11[0][0]    │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 474,049 (1.81 MB)

 Trainable params: 474,049 (1.81 MB)

 Non-trainable params: 0 (0.00 B)

In [68]:
cp2 = ModelCheckpoint('Photovoltaic_model.keras', save_best_only=True)
Photo_model.compile(optimizer=Adam(learning_rate=0.0001), loss="mean_squared_error", metrics=['mae'])
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

In [69]:
history = Photo_model.fit(
    X_train_windowed, y_train_windowed,
    validation_data=(X_val_windowed, y_val_windowed),
    epochs=50,
    batch_size=16,
    callbacks=[cp, early_stop]
)

Epoch 1/50


226/226 ━━━━━━━━━━━━━━━━━━━━ 363s 344ms/step - loss: 1.0836 - mae: 0.2751 - val_loss: 0.5936 - val_mae: 0.3221
Epoch 2/50
226/226 ━━━━━━━━━━━━━━━━━━━━ 62s 225ms/step - loss: 0.4806 - mae: 0.2846 - val_loss: 0.2998 - val_mae: 0.3024
Epoch 3/50
226/226 ━━━━━━━━━━━━━━━━━━━━ 74s 172ms/step - loss: 0.2146 - mae: 0.2155 - val_loss: 0.1419 - val_mae: 0.1931
Epoch 4/50
226/226 ━━━━━━━━━━━━━━━━━━━━ 99s 221ms/step - loss: 0.1102 - mae: 0.1638 - val_loss: 0.0987 - val_mae: 0.1856
Epoch 5/50
226/226 ━━━━━━━━━━━━━━━━━━━━ 78s 178ms/step - loss: 0.0768 - mae: 0.1583 - val_loss: 0.0881 - val_mae: 0.1861
Epoch 6/50
226/226 ━━━━━━━━━━━━━━━━━━━━ 99s 229ms/step - loss: 0.0590 - mae: 0.1493 - val_loss: 0.1052 - val_mae: 0.2102
Epoch 7/50
226/226 ━━━━━━━━━━━━━━━━━━━━ 56s 218ms/step - loss: 0.0588 - mae: 0.1535 - val_loss: 0.0770 - val_mae: 0.1914
Epoch 8/50
226/226 ━━━━━━━━━━━━━━━━━━━━ 47s 179ms/step - loss: 0.0513 - mae: 0.1474 - val_loss: 0.0707 - val_mae: 0.1796
Epoch 9/50
226/226 ━━━━━━━━━━━━━━━━━━━━ 87

In [70]:
Photo_predictions = Photo_model.predict(X_test_windowed)
Photo_predictions

23/23 ━━━━━━━━━━━━━━━━━━━━ 35s 866ms/step


array([[ 7.04868790e-03],
       [ 4.04664166e-02],
       [ 2.44134456e-01],
       [ 7.06092000e-01],
       [ 6.08939528e-01],
       [ 3.84140909e-01],
       [ 1.83531314e-01],
       [ 2.94742107e-01],
       [ 6.31554961e-01],
       [ 7.09556103e-01],
       [ 4.70561057e-01],
       [ 4.47456568e-01],
       [ 1.37081459e-01],
       [ 4.49688464e-01],
       [ 8.00388515e-01],
       [ 1.04231822e+00],
       [ 1.07957542e+00],
       [ 8.46226037e-01],
       [ 7.23464251e-01],
       [ 1.49178818e-01],
       [ 1.43082310e-02],
       [ 9.58504248e-03],
       [ 1.81454569e-01],
       [ 3.60414654e-01],
       [ 8.27250302e-01],
       [ 8.96868050e-01],
       [ 9.22353804e-01],
       [ 8.01602960e-01],
       [ 2.70325728e-02],
       [ 2.41547897e-02],
       [ 1.32010013e-01],
       [ 3.14393729e-01],
       [ 7.59261966e-01],
       [ 6.50186896e-01],
       [ 4.12922680e-01],
       [ 4.88438636e-01],
       [ 7.52671540e-01],
       [ 7.65576363e-01],
       [ 6.3

In [71]:
Photo_predictions = y_scaler.inverse_transform(Photo_predictions.reshape(-1, 1))
Photo_predictions = np.clip(Photo_predictions, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000

In [72]:
Photo_resultados = pd.DataFrame(Photo_predictions, index = y_test_windowed.index, columns=["Photo"])

In [73]:
predicciones["Photo"] = Photo_resultados["Photo"]
predicciones

,Generación,LightGBM,Random Forest,CTNET,Forecast,Photo
16400,642.0,1485.421686,1850.979413,NaN,NaN,NaN
16401,6804.0,11718.606066,11182.089488,NaN,NaN,NaN
16402,10501.0,18614.042112,19279.054526,NaN,NaN,NaN
16403,13147.0,17689.00615,17438.807493,NaN,NaN,NaN
16404,15536.0,17039.389972,16974.916203,NaN,NaN,NaN
...,...,...,...,...,...,...
18273,7356.0,6900.654023,5846.106045,712.266174,11134.365234,10667.417969
18274,17638.0,20543.60541,19516.687678,16165.256836,17797.712891,14641.859375
18275,23339.0,23189.371101,21231.472263,20088.689453,18955.470703,18846.126953
18276,26323.0,26104.467998,24456.544978,24133.009766,14881.110352,12324.421875


In [74]:
print("LightGBM")
print(f"MAE: {mean_absolute_error(predicciones['Generación'], predicciones['LightGBM']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones['Generación'], predicciones['LightGBM'])):.4f}")
print(f"R²: {r2_score(predicciones['Generación'], predicciones['LightGBM']):.4f}")
print("Random Forest")
print(f"MAE: {mean_absolute_error(predicciones['Generación'], predicciones['Random Forest']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones['Generación'], predicciones['Random Forest'])):.4f}")
print(f"R²: {r2_score(predicciones['Generación'], predicciones['Random Forest']):.4f}")
print("CTNET")
print(f"MAE: {mean_absolute_error(predicciones.loc[~predicciones['CTNET'].isna(),'Generación'], predicciones.loc[~predicciones['CTNET'].isna(),'CTNET']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones.loc[~predicciones['CTNET'].isna(),'Generación'], predicciones.loc[~predicciones['CTNET'].isna(),'CTNET'])):.4f}")
print(f"R²: {r2_score(predicciones.loc[~predicciones['CTNET'].isna(),'Generación'], predicciones.loc[~predicciones['CTNET'].isna(),'CTNET']):.4f}")
print("Forecast")
print(f"MAE: {mean_absolute_error(predicciones.loc[~predicciones['Forecast'].isna(),'Generación'], predicciones.loc[~predicciones['Forecast'].isna(),'Forecast']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones.loc[~predicciones['Forecast'].isna(),'Generación'], predicciones.loc[~predicciones['Forecast'].isna(),'Forecast'])):.4f}")
print(f"R²: {r2_score(predicciones.loc[~predicciones['Forecast'].isna(),'Generación'], predicciones.loc[~predicciones['Forecast'].isna(),'Forecast']):.4f}")
print("Photovoltaic")
print(f"MAE: {mean_absolute_error(predicciones.loc[~predicciones['Photo'].isna(),'Generación'], predicciones.loc[~predicciones['Photo'].isna(),'Photo']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones.loc[~predicciones['Photo'].isna(),'Generación'], predicciones.loc[~predicciones['Photo'].isna(),'Photo'])):.4f}")
print(f"R²: {r2_score(predicciones.loc[~predicciones['Photo'].isna(),'Generación'], predicciones.loc[~predicciones['Photo'].isna(),'Photo']):.4f}")

LightGBM
MAE: 1424.1067
RMSE: 2418.9925
R²: 0.9503
Random Forest
MAE: 1391.4381
RMSE: 2497.2550
R²: 0.9470
CTNET
MAE: 4856.1916
RMSE: 6664.5809
R²: 0.6342
Forecast
MAE: 4399.2846
RMSE: 6111.5054
R²: 0.6924
Photovoltaic
MAE: 4019.0234
RMSE: 5994.1247
R²: 0.7041


In [75]:
# Seleccionar las columnas desde "LightGBM" en adelante
columnas_nuevas = predicciones.loc[:, "LightGBM":]

# Unir con `datos` usando el índice, manteniendo todo en `datos`
datos = datos.merge(columnas_nuevas, left_index=True, right_index=True, how='left')

# Ver resultado
datos.head()


,Fecha,Generación,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Cluster KMeans,Cluster GMM,Generacion_prev_hour,Generacion_prev_day,LightGBM,Random Forest,CTNET,Forecast,Photo
24,2022-09-02 00:00:00,0.0,19,6,76,0,4,15,0,Noche,Noche,0.0,0.0,NaN,NaN,NaN,NaN,NaN
25,2022-09-02 01:00:00,0.0,18,7,81,0,4,15,1,Noche,Noche,0.0,0.0,NaN,NaN,NaN,NaN,NaN
26,2022-09-02 02:00:00,0.0,18,7,84,0,4,15,2,Noche,Noche,0.0,0.0,NaN,NaN,NaN,NaN,NaN
27,2022-09-02 03:00:00,0.0,18,7,86,0,4,15,3,Noche,Noche,0.0,0.0,NaN,NaN,NaN,NaN,NaN
28,2022-09-02 04:00:00,0.0,17,7,86,0,4,15,4,Noche,Noche,0.0,0.0,NaN,NaN,NaN,NaN,NaN


## X_train para hacer análisis de sobreajuste

In [76]:
predicciones_train = y_train.copy()

In [77]:
LightGBM_predictions_train = LightGBM_model.predict(X_train_scaled_df)
LightGBM_predictions_train = y_scaler.inverse_transform(LightGBM_predictions_train.reshape(-1, 1))
LightGBM_predictions_train = np.clip(LightGBM_predictions_train, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000
LightGBM_resultados = pd.DataFrame(LightGBM_predictions_train, index = y_train_scaled_df.index, columns=["LightGBM_train"])
predicciones_train["LightGBM_train"] = LightGBM_resultados["LightGBM_train"]

[LightGBM] [Warning] min_data_in_leaf is set=85, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=85


In [78]:
RandomForest_predictions_train = RF_model.predict(X_train_scaled_df)
RandomForest_predictions_train = y_scaler.inverse_transform(RandomForest_predictions_train.reshape(-1, 1))
RandomForest_predictions_train = np.clip(RandomForest_predictions_train, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000
RandomForest_resultados = pd.DataFrame(RandomForest_predictions_train, index = y_train_scaled_df.index, columns=["RandomForest_train"])
predicciones_train["RandomForest_train"] = RandomForest_resultados["RandomForest_train"]

In [79]:
CTNET_predictions_train = CTNET.predict(X_train_windowed)
CTNET_predictions_train = y_scaler.inverse_transform(CTNET_predictions_train.reshape(-1, 1))
CTNET_predictions_train = np.clip(CTNET_predictions_train, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000
CTNET_resultados = pd.DataFrame(CTNET_predictions_train, index = y_train_windowed.index, columns=["CTNET_train"])
predicciones_train["CTNET_train"] = CTNET_resultados["CTNET_train"]

113/113 ━━━━━━━━━━━━━━━━━━━━ 19s 94ms/step


In [80]:
Forecast_predictions_train = Forecast_model.predict(X_train_windowed)
Forecast_predictions_train = y_scaler.inverse_transform(Forecast_predictions_train.reshape(-1, 1))
Forecast_predictions_train = np.clip(Forecast_predictions_train, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000
Forecast_resultados = pd.DataFrame(Forecast_predictions_train, index = y_train_windowed.index, columns=["Forecast_train"])
predicciones_train["Forecast_train"] = Forecast_resultados["Forecast_train"]

113/113 ━━━━━━━━━━━━━━━━━━━━ 21s 152ms/step


In [81]:
Photo_predictions_train = Photo_model.predict(X_train_windowed)
Photo_predictions_train = y_scaler.inverse_transform(Photo_predictions_train.reshape(-1, 1))
Photo_predictions_train = np.clip(Photo_predictions_train, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000
Photo_resultados = pd.DataFrame(Photo_predictions_train, index = y_train_windowed.index, columns=["Photo_train"])
predicciones_train["Photo_train"] = Photo_resultados["Photo_train"]

113/113 ━━━━━━━━━━━━━━━━━━━━ 12s 59ms/step


In [82]:
predicciones_train

,Generación,LightGBM_train,RandomForest_train,CTNET_train,Forecast_train,Photo_train
30,0.000000,38.404720,0.000000,NaN,NaN,NaN
31,0.000000,38.243795,33.638347,NaN,NaN,NaN
32,438.814997,743.496439,708.929118,NaN,NaN,NaN
33,5908.000884,9112.736385,6958.973311,NaN,NaN,NaN
34,5030.740421,9605.211233,6928.488764,NaN,NaN,NaN
...,...,...,...,...,...,...
12958,0.000000,0.000000,0.000000,337.538696,0.000000,746.106812
12967,0.000000,86.816623,0.000000,284.666016,0.000000,425.357269
12968,0.000000,50.108601,0.000000,350.532196,0.000000,672.581909
12969,295.000000,335.095107,282.147500,890.763672,0.000000,1797.385864


In [83]:
print("LightGBM")
print(f"MAE: {mean_absolute_error(predicciones_train['Generación'], predicciones_train['LightGBM_train']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones_train['Generación'], predicciones_train['LightGBM_train'])):.4f}")
print(f"R²: {r2_score(predicciones_train['Generación'], predicciones_train['LightGBM_train']):.4f}")
print("Random Forest")
print(f"MAE: {mean_absolute_error(predicciones_train['Generación'], predicciones_train['RandomForest_train']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones_train['Generación'], predicciones_train['RandomForest_train'])):.4f}")
print(f"R²: {r2_score(predicciones_train['Generación'], predicciones_train['RandomForest_train']):.4f}")
print("CTNET")
print(f"MAE: {mean_absolute_error(predicciones_train.loc[~predicciones_train['CTNET_train'].isna(),'Generación'], predicciones_train.loc[~predicciones_train['CTNET_train'].isna(),'CTNET_train']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones_train.loc[~predicciones_train['CTNET_train'].isna(),'Generación'], predicciones_train.loc[~predicciones_train['CTNET_train'].isna(),'CTNET_train'])):.4f}")
print(f"R²: {r2_score(predicciones_train.loc[~predicciones_train['CTNET_train'].isna(),'Generación'], predicciones_train.loc[~predicciones_train['CTNET_train'].isna(),'CTNET_train']):.4f}")
print("Forecast")
print(f"MAE: {mean_absolute_error(predicciones_train.loc[~predicciones_train['Forecast_train'].isna(),'Generación'], predicciones_train.loc[~predicciones_train['Forecast_train'].isna(),'Forecast_train']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones_train.loc[~predicciones_train['Forecast_train'].isna(),'Generación'], predicciones_train.loc[~predicciones_train['Forecast_train'].isna(),'Forecast_train'])):.4f}")
print(f"R²: {r2_score(predicciones_train.loc[~predicciones_train['Forecast_train'].isna(),'Generación'], predicciones_train.loc[~predicciones_train['Forecast_train'].isna(),'Forecast_train']):.4f}")
print("Photovoltaic")
print(f"MAE: {mean_absolute_error(predicciones_train.loc[~predicciones_train['Photo_train'].isna(),'Generación'], predicciones_train.loc[~predicciones_train['Photo_train'].isna(),'Photo_train']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones_train.loc[~predicciones_train['Photo_train'].isna(),'Generación'], predicciones_train.loc[~predicciones_train['Photo_train'].isna(),'Photo_train'])):.4f}")
print(f"R²: {r2_score(predicciones_train.loc[~predicciones_train['Photo_train'].isna(),'Generación'], predicciones_train.loc[~predicciones_train['Photo_train'].isna(),'Photo_train']):.4f}")

LightGBM
MAE: 1192.3439
RMSE: 2165.4793
R²: 0.9500
Random Forest
MAE: 503.4792
RMSE: 979.1845
R²: 0.9898
CTNET
MAE: 3696.0014
RMSE: 5815.7206
R²: 0.6368
Forecast
MAE: 3715.3918
RMSE: 5986.6397
R²: 0.6151
Photovoltaic
MAE: 3427.5738
RMSE: 5252.4921
R²: 0.7037


In [84]:
# Seleccionar las columnas desde "LightGBM" en adelante
columnas_nuevas = predicciones_train.loc[:, "LightGBM_train":]

# Unir con `datos` usando el índice, manteniendo todo en `datos`
datos = datos.merge(columnas_nuevas, left_index=True, right_index=True, how='left')

# Ver resultado
datos.head()


,Fecha,Generación,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Cluster KMeans,...,LightGBM,Random Forest,CTNET,Forecast,Photo,LightGBM_train,RandomForest_train,CTNET_train,Forecast_train,Photo_train
24,2022-09-02 00:00:00,0.0,19,6,76,0,4,15,0,Noche,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25,2022-09-02 01:00:00,0.0,18,7,81,0,4,15,1,Noche,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
26,2022-09-02 02:00:00,0.0,18,7,84,0,4,15,2,Noche,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
27,2022-09-02 03:00:00,0.0,18,7,86,0,4,15,3,Noche,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
28,2022-09-02 04:00:00,0.0,17,7,86,0,4,15,4,Noche,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [85]:
datos.to_excel("04.3_Predicciones_Conjunto_nublado KMeans.xlsx", index=True)

## Guardamos los modelos

In [86]:
import joblib

# Guardar modelo LightGBM
joblib.dump(LightGBM_model, "4_3_LightGBM_model.pkl")

# Guardar modelo Random Forest
joblib.dump(RF_model, "4_3_RandomForest_model.pkl")


['4_3_RandomForest_model.pkl']

In [87]:
CTNET.save("4_3_CTNET_model.keras")
Forecast_model.save("4_3_Forecast_model.keras")
Photo_model.save("4_3_Photo_model.keras")